# Download the data from the Don'tGetKicked contest.

In [57]:
import pandas as pd
import numpy as np
from category_encoders import CountEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.neural_network import MLPClassifier
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

In [58]:
df = pd.read_csv(filepath_or_buffer='data/training.csv', index_col=0, parse_dates=['PurchDate'])
df['PRIMEUNIT'] = df['PRIMEUNIT'].fillna('MISSING')
df['AUCGUART'] = df['AUCGUART'].fillna('MISSING')
df.isna().sum()

IsBadBuy                                0
PurchDate                               0
Auction                                 0
VehYear                                 0
VehicleAge                              0
Make                                    0
Model                                   0
Trim                                 2360
SubModel                                8
Color                                   8
Transmission                            9
WheelTypeID                          3169
WheelType                            3174
VehOdo                                  0
Nationality                             5
Size                                    5
TopThreeAmericanName                    5
MMRAcquisitionAuctionAveragePrice      18
MMRAcquisitionAuctionCleanPrice        18
MMRAcquisitionRetailAveragePrice       18
MMRAcquisitonRetailCleanPrice          18
MMRCurrentAuctionAveragePrice         315
MMRCurrentAuctionCleanPrice           315
MMRCurrentRetailAveragePrice      

In [59]:
y = df['IsBadBuy']
X = df[df.columns[1:]]
X

,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,Color,Transmission,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
RefId,,,,,,,,,,,,,,,,,,,,,
1,2009-12-07,ADESA,2006,3,MAZDA,MAZDA3,i,4D SEDAN I,RED,AUTO,...,11597.0,12409.0,MISSING,MISSING,21973,33619,FL,7100.0,0,1113
2,2009-12-07,ADESA,2004,5,DODGE,1500 RAM PICKUP 2WD,ST,QUAD CAB 4.7L SLT,WHITE,AUTO,...,11374.0,12791.0,MISSING,MISSING,19638,33619,FL,7600.0,0,1053
3,2009-12-07,ADESA,2005,4,DODGE,STRATUS V6,SXT,4D SEDAN SXT FFV,MAROON,AUTO,...,7146.0,8702.0,MISSING,MISSING,19638,33619,FL,4900.0,0,1389
4,2009-12-07,ADESA,2004,5,DODGE,NEON,SXT,4D SEDAN,SILVER,AUTO,...,4375.0,5518.0,MISSING,MISSING,19638,33619,FL,4100.0,0,630
5,2009-12-07,ADESA,2005,4,FORD,FOCUS,ZX3,2D COUPE ZX3,SILVER,MANUAL,...,6739.0,7911.0,MISSING,MISSING,19638,33619,FL,4000.0,0,1020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73010,2009-12-02,ADESA,2001,8,MERCURY,SABLE,GS,4D SEDAN GS,BLACK,AUTO,...,4836.0,5937.0,MISSING,MISSING,18111,30212,GA,4200.0,0,993
73011,2009-12-02,ADESA,2007,2,CHEVROLET,MALIBU 4C,LS,4D SEDAN LS,SILVER,AUTO,...,10151.0,11652.0,MISSING,MISSING,18881,30212,GA,6200.0,0,1038
73012,2009-12-02,ADESA,2005,4,JEEP,GRAND CHEROKEE 2WD V,Lar,4D WAGON LAREDO,SILVER,AUTO,...,11831.0,14402.0,MISSING,MISSING,18111,30212,GA,8200.0,0,1893


In [60]:
def my_train_valid_test_date_split(X, y, validation_date, test_date):
    train_mask = X['PurchDate'] < validation_date
    valid_mask = (X['PurchDate'] >= validation_date) & (X['PurchDate'] <= test_date)
    test_mask = X['PurchDate'] > test_date

    X_train = X[train_mask]
    X_valid = X[valid_mask]
    X_test = X[test_mask]

    y_train = y[train_mask]
    y_valid = y[valid_mask]
    y_test = y[test_mask]

    X_train = X_train.drop(labels=['PurchDate'], axis=1)
    X_valid = X_valid.drop(labels=['PurchDate'], axis=1)
    X_test = X_test.drop(labels=['PurchDate'], axis=1)

    return X_train, X_valid, X_test, y_train, y_valid, y_test

In [61]:
first_date = df['PurchDate'].quantile(0.34)
second_date = df['PurchDate'].quantile(0.67)
print(first_date)
print(second_date)

2009-09-17 00:00:00
2010-05-18 00:00:00


In [62]:
X_train, X_valid, X_test, y_train, y_valid, y_test = my_train_valid_test_date_split(X, y, first_date, second_date)
print(len(X_train), len(X_valid), len(X_test))

24695 24372 23916


In [63]:
fill_values = {
    'WheelType':    X_train['WheelType'].mode()[0],
    'WheelTypeID':  X_train['WheelTypeID'].mode()[0],
    'Trim':         X_train['Trim'].mode()[0],
    'SubModel':     X_train['SubModel'].mode()[0],
    'Color':        X_train['Color'].mode()[0],
    'Transmission': X_train['Transmission'].mode()[0],
    'Nationality':  X_train['Nationality'].mode()[0],
    'Size':         X_train['Size'].mode()[0],
    'TopThreeAmericanName': X_train['TopThreeAmericanName'].mode()[0],
    'MMRAcquisitionAuctionAveragePrice': X_train['MMRAcquisitionAuctionAveragePrice'].median(),
    'MMRAcquisitionAuctionCleanPrice':   X_train['MMRAcquisitionAuctionCleanPrice'].median(),
    'MMRAcquisitionRetailAveragePrice':  X_train['MMRAcquisitionRetailAveragePrice'].median(),
    'MMRAcquisitonRetailCleanPrice':     X_train['MMRAcquisitonRetailCleanPrice'].median(),
    'MMRCurrentAuctionAveragePrice':     X_train['MMRCurrentAuctionAveragePrice'].median(),
    'MMRCurrentAuctionCleanPrice':       X_train['MMRCurrentAuctionCleanPrice'].median(),
    'MMRCurrentRetailAveragePrice':      X_train['MMRCurrentRetailAveragePrice'].median(),
    'MMRCurrentRetailCleanPrice':        X_train['MMRCurrentRetailCleanPrice'].median(),
}

X_train.fillna(fill_values, inplace=True)
X_valid.fillna(fill_values, inplace=True)
X_test.fillna(fill_values, inplace=True)

In [64]:
categorical_cols = [
    'Auction', 'Make', 'Model', 'Trim', 'SubModel', 'Color',
    'Transmission', 'WheelType', 'Nationality', 'Size',
    'TopThreeAmericanName', 'PRIMEUNIT', 'AUCGUART', 'VNST',
    'WheelTypeID', 'BYRNO', 'VNZIP1'
]

In [65]:
encoder = CountEncoder(cols=categorical_cols)
X_train = encoder.fit_transform(X_train)
X_valid = encoder.transform(X_valid)
X_test = encoder.transform(X_test)
X_test

,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,Color,Transmission,WheelTypeID,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
RefId,,,,,,,,,,,,,,,,,,,,,
266,5076,2006,4,5743.0,0.0,3039.0,142.0,109,24008,10788.0,...,12427.0,13850.0,24693.0,24693.0,545.0,458.0,3452.0,6800.0,0,1630
267,5076,2004,6,5743.0,0.0,183.0,493.0,1921,24008,13666.0,...,9497.0,11161.0,24693.0,24693.0,913.0,458.0,3452.0,8600.0,0,1020
268,5076,2006,4,2832.0,3.0,5359.0,4991.0,2048,24008,10788.0,...,8667.0,9390.0,24693.0,24693.0,913.0,458.0,3452.0,6150.0,0,1215
269,5076,2007,3,656.0,9.0,210.0,179.0,729,24008,10788.0,...,9792.0,10587.0,24693.0,24693.0,913.0,458.0,3452.0,5800.0,0,728
270,5076,2002,8,287.0,1.0,233.0,106.0,2048,24008,13666.0,...,8950.0,10403.0,24693.0,24693.0,545.0,458.0,3452.0,7600.0,0,1543
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72415,5076,2004,6,656.0,0.0,47.0,47.0,2504,24008,241.0,...,6364.0,7176.0,24693.0,24693.0,0.0,409.0,4615.0,4105.0,0,2891
72416,5076,2003,7,4852.0,2.0,434.0,12.0,2048,24008,13666.0,...,7853.0,8917.0,0.0,0.0,0.0,409.0,4615.0,4490.0,0,1930
72417,5076,2002,8,4323.0,2.0,3484.0,1448.0,3486,24008,13666.0,...,5411.0,6523.0,0.0,0.0,0.0,409.0,4615.0,3690.0,0,1455


In [66]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
X_test = scaler.transform(X_test)

# Create a Python class, MLP with 1 hidden layer and sigmoid activation function at the end of the network. It should support fit, predict_proba and predict methods. Also, the number of neurons in the hidden layer and the activation function must be parameters of your class.

In [67]:
def sigmoid(X):
    return 1 / (1 + np.exp(-X))

In [68]:
def relu(z):
   return np.maximum(0, z)

In [69]:
class myMLP:
    def __init__(self, n_hidden = 100, activation=sigmoid, epochs=100, learning_rate=0.01, random_state=21):
        self.n_hidden = n_hidden
        self.activation = activation
        self.batch_size = 32
        self.epochs = epochs
        self.learning_rate = learning_rate
        self.np_rng = np.random.default_rng(random_state)
        self.activation_derivatives = {
            np.arctan: lambda x: 1 / (1 + x**2),
            np.tanh: lambda x: 1 - np.tanh(x)**2,
            sigmoid: lambda x: sigmoid(x) * (1 - sigmoid(x)),
            relu: lambda x: (x > 0).astype(float)
        }

    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y)
        d = X.shape[1]
        self.weights1 = self.np_rng.normal(0.0, 1/np.sqrt(d), size=(d, self.n_hidden))
        self.bias1 = np.zeros(self.n_hidden)
        self.weights2 = self.np_rng.normal(0.0, 1/np.sqrt(self.n_hidden), size=self.n_hidden)
        self.bias2 = 0
        self.history = []
        for epoch in range(self.epochs):
            indexes = self.np_rng.permutation(len(X))
            batch_loses = []
            for i in range(0, len(indexes), self.batch_size):
                b = indexes[i:i+self.batch_size]
                xi, yi = X[b], y[b]
                z1 = self.bias1 + (xi @ self.weights1)
                a1 = self.activation(z1)
                z2 = self.bias2 + (a1 @ self.weights2)
                a2 = sigmoid(z2)
                p = np.clip(a2, 1e-12, 1 - 1e-12)
                loss = - np.mean(yi * np.log(p) + (1 - yi) * np.log(1 - p))
                batch_loses.append(loss)
                d2 = (a2 - yi)/len(xi) #dl / dz2
                gradLdw2 = d2 @ a1 #dl / dw2
                gradLb2 = np.sum(d2) #dl / db2
                gradLda1 = d2[:, None] * self.weights2[None, :]
                d1 = gradLda1 * self.activation_derivatives[self.activation](z1) #dl / dz1
                gradLdw1 = xi.T @ d1
                gradLb1 = np.sum(d1, axis=0)
                self.weights1 -= self.learning_rate * gradLdw1
                self.bias1 -= self.learning_rate * gradLb1
                self.weights2 -= self.learning_rate * gradLdw2
                self.bias2 -= self.learning_rate * gradLb2
            epoch_loss = np.mean(batch_loses)
            self.history.append(epoch_loss)

    def predict_proba(self, X):
        X = np.array(X)
        z1 = self.bias1 + X @ self.weights1
        a1 = self.activation(z1)
        z2 = self.bias2 + a1 @ self.weights2
        a2 = sigmoid(z2)
        return a2

    def predict(self, X):
        p = self.predict_proba(X)
        return (p >= 0.5).astype(int)

# With your MLP module and careful network engineering, you must obtain at least a 0.15 Gini score on the validation dataset. You can train for more than 1 epoch, use different activation functions, and different optimizers (like SGD or Adam).

In [70]:
perec = myMLP(n_hidden=100, activation=np.arctan, epochs=100, learning_rate=0.01, random_state=21)
perec.fit(X_train, y_train)
gini = 2 * roc_auc_score(y_valid, perec.predict_proba(X_valid)) - 1
gini

0.33910339958727054

# Use sklearn's MLPClassifier and check its performance on the validation dataset. Is it better than your module? If so, why?

In [71]:
skMLP = MLPClassifier(hidden_layer_sizes=(100,), activation='tanh', random_state=21)
skMLP.fit(X_train, y_train)
gini = 2 * roc_auc_score(y_valid, skMLP.predict_proba(X_valid)[:, 1]) - 1
gini

C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


0.23484532887829457

# Implement and try different activation functions: sigmoid, ReLU, cosine. Remember that you have to derive gradients for each different activation function. Which activation function gives the best performance on the validation dataset?

In [72]:
myMLPrelu = myMLP(n_hidden=100, activation=relu, epochs=100, learning_rate=0.01, random_state=21)
myMLPrelu.fit(X_train, y_train)
gini_relu = 2 * roc_auc_score(y_valid, myMLPrelu.predict_proba(X_valid)) - 1
gini_relu

0.3392946186301804

In [73]:
myMLPtanh = myMLP(n_hidden=100, activation=np.tanh, epochs=100, learning_rate=0.01, random_state=21)
myMLPtanh.fit(X_train, y_train)
gini_tanh = 2 * roc_auc_score(y_valid, myMLPtanh.predict_proba(X_valid)) - 1
gini_tanh

0.3339444899380526

In [74]:
myMLPsigma = myMLP(n_hidden=100, activation=sigmoid, epochs=100, learning_rate=0.01, random_state=21)
myMLPsigma.fit(X_train, y_train)
gini_sigma = 2 * roc_auc_score(y_valid, myMLPsigma.predict_proba(X_valid)) - 1
gini_sigma

0.34122150218464586

##### relu is the best

# Design an MLP module with 1 hidden layer using any high level deep learning framework: Pytorch, Keras, or Tensorflow. Check its performance on the validation dataset. Is it better than your module? If so, why?

In [75]:
Xtrain = torch.tensor(np.asarray(X_train), dtype=torch.float32)
ytrain = torch.tensor(np.asarray(y_train), dtype=torch.float32)
Xvalid = torch.tensor(np.asarray(X_valid), dtype=torch.float32)
yvalid = torch.tensor(np.asarray(y_valid), dtype=torch.float32)
Xtest = torch.tensor(np.asarray(X_test), dtype=torch.float32)
ytest = torch.tensor(np.asarray(y_test), dtype=torch.float32)

In [76]:
train_ds = TensorDataset(Xtrain, ytrain)
train_loader = DataLoader(dataset=train_ds, batch_size=32, shuffle=True)

In [77]:
model = nn.Sequential(
    nn.Linear(Xtrain.shape[1], 100),
    nn.Tanh(),
    nn.Linear(100, 1)
)

In [78]:
loss_fun = nn.BCEWithLogitsLoss()

In [79]:
opt = torch.optim.SGD(model.parameters(), lr=0.01)

In [80]:
for epoch in range(100):
    epoch_losses = []
    for xb, yb in train_loader:
        opt.zero_grad()
        out = model(xb).squeeze(1)
        loss = loss_fun(out, yb)
        loss.backward()
        opt.step()
        epoch_losses.append(loss.item())
    print(np.mean(epoch_losses), epoch)

0.40037570548258294 0
0.33642230870038115 1
0.3343525340616549 2
0.3337699709257468 3
0.33345544359458545 4
0.33299059139014525 5
0.3330383925587711 6
0.332717647216249 7
0.3326503071895381 8
0.3324815191791774 9
0.33223655153973114 10
0.3322042290970606 11
0.3319590962612567 12
0.33193568999208317 13
0.3317689813596288 14
0.3316876571528961 15
0.33152416020322956 16
0.3313180556704189 17
0.33116936701991706 18
0.3312214364929829 19
0.3309919645845736 20
0.33084455357352993 21
0.3306655185152366 22
0.33091063542687216 23
0.33059421650717913 24
0.33061620610442805 25
0.3305103851414715 26
0.33035650493220037 27
0.33033595285306017 28
0.3301897106759752 29
0.33002707994316216 30
0.33014185201627605 31
0.3297689990598921 32
0.3298272791323421 33
0.3297897539550313 34
0.329602096519322 35
0.3296009626123726 36
0.3295377473409621 37
0.3293914352036974 38
0.3293527662657549 39
0.32928040979787165 40
0.32908863256748166 41
0.3291131855697984 42
0.3288935485532401 43
0.3289474958814488 44
0.32

In [81]:
model.eval()
with torch.no_grad():
    p = torch.sigmoid(model(Xvalid).squeeze(1))
gini = 2 * roc_auc_score(yvalid.numpy(), p.numpy()) - 1
print(gini)

0.3609750768865796


# Take the best model and estimate its performance on the test dataset: check the Gini scores on all three datasets for your best model: training Gini, valid Gini, test Gini. Do you see a drop in performance when comparing the valid quality to the test quality? Is your model overfitting or not? Explain.


In [85]:
gini_relu_train = 2 * roc_auc_score(y_train, myMLPrelu.predict_proba(X_train)) - 1
gini_relu_train

0.4908584428096834

In [86]:
gini_relu_valid = gini_relu
gini_relu_valid

0.3392946186301804

In [87]:
gini_relu_test = 2 * roc_auc_score(y_test, myMLPrelu.predict_proba(X_test)) - 1
gini_relu_test

0.14956860190835797